# Exploración del Espacio Químico del Dataset QM9
## Reducción de Dimensionalidad, Búsqueda de Similitud y Mapas Topológicos

---

### Contexto del proyecto

El dataset **QM9** contiene ~134,000 moléculas pequeñas compuestas por carbono, hidrógeno, oxígeno, nitrógeno y flúor. Cada molécula está representada mediante su cadena **SMILES** (*Simplified Molecular Input Line Entry System*), una notación textual que codifica la estructura química.

El objetivo de este notebook es construir y explorar un **espacio vectorial de moléculas**: una representación matemática donde moléculas estructuralmente similares se encuentran geométricamente cercanas. Para ello se recorre el siguiente pipeline:

```
Moléculas (SMILES)
        │
        ▼
   mol2vec + model_300dim.pkl
        │  Embedding semántico de subestructuras
        ▼
  Vectores de 300 dimensiones
        │
        ├──► Búsqueda de similitud (Mahalanobis, 300D)
        │
        ▼
    Autoencoder (PyTorch)
        │  Compresión no lineal
        ▼
  Vectores de 32 dimensiones (espacio latente)
        │
        ├──► Búsqueda de similitud (Mahalanobis, 32D)
        │
        ▼
  Descomposición de Cholesky
        │  Transforma el espacio para que Euclidiana = Mahalanobis
        ▼
  Espacio latente optimizado
        │
        ├──► Chupón Gaussiano (vecindad topológica)
        │
        ▼
  UMAP / t-SNE  ──►  HDBSCAN
        │              │
        ▼              ▼
   Mapa 2D      Clusters químicos
```

---

### Estructura de carpetas esperada

```
proyecto/
├── notebooks/
│   └── Similitud_v2.ipynb       ← este archivo
├── data/
│   ├── qm9.csv                   ← dataset descargado
│   ├── qm9_embeddings_flat.csv   ← embeddings 300D (smiles + 300 columnas)
│   ├── qm9_embeddings.npy        ← embeddings 300D como matriz numpy
│   ├── qm9_latent.npy            ← espacio latente 32D del autoencoder
│   ├── qm9_latent_mahalanobis_space.npy  ← espacio transformado con Cholesky
│   ├── qm9_umap_2d.npy           ← coordenadas UMAP (se genera una sola vez)
│   └── qm9_tsne_2d.npy           ← coordenadas t-SNE (se genera una sola vez)
├── models/
│   └── model_300dim.pkl          ← modelo mol2vec preentrenado
├── html/
│   ├── mapa_topologico_HDBSCAN_UMAP.html
│   ├── mapa_topologico_HDBSCAN_T-SNE.html
│   ├── mapa_chupon_umap.html
│   └── mapa_chupon_tsne.html
├── .devcontainer/
│   └── devcontainer.json
├── requirements.txt
├── environment.yml
└── README.md
```

---
## 0. Configuración del entorno

**Motivación:** El notebook debe ejecutarse de forma idéntica en dos entornos distintos: Linux local y Google Colab. La diferencia principal está en la ubicación del directorio raíz del proyecto y en si es necesario instalar dependencias.

Esta celda detecta automáticamente el entorno y configura la variable `BASE_DIR`, que todas las secciones posteriores utilizan como punto de referencia para leer y escribir archivos. De esta manera, ninguna ruta queda hardcodeada.

In [ ]:
import os
import sys

# Detectar si estamos en Google Colab
EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    # En Colab, montamos Drive y clonamos/subimos el proyecto ahí
    from google.colab import drive
    drive.mount('/content/drive')
    # Ajusta esta ruta a donde tengas la carpeta del proyecto en tu Drive
    BASE_DIR = '/content/drive/MyDrive/proyecto'
else:
    # En local, el notebook vive en /proyecto/notebooks/
    # BASE_DIR apunta a la raíz del proyecto (un nivel arriba)
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Crear directorios necesarios si no existen
for subdir in ['data', 'models', 'html']:
    os.makedirs(os.path.join(BASE_DIR, subdir), exist_ok=True)

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Linux local'}")
print(f"Directorio raíz del proyecto: {BASE_DIR}")

### 0.1 Instalación de dependencias

**Motivación:** `mol2vec` es una librería que dejó de recibir actualizaciones activas. No se encuentra en PyPI de forma directa en todas las versiones de Python, por lo que se instala desde su repositorio en GitHub. El resto de dependencias sí están disponibles con versiones estables.

Si ya tienes el entorno configurado (ya sea con `requirements.txt` o `environment.yml`), puedes saltar esta celda.

In [ ]:
# Solo ejecutar si no tienes el entorno preinstalado
# Para evitar conflictos, se recomienda usar environment.yml o requirements.txt
# (ver README.md para instrucciones)

# mol2vec se instala directamente desde GitHub porque su versión en PyPI
# puede no resolver dependencias correctamente en Python >= 3.8
!pip install git+https://github.com/samoturk/mol2vec

# Resto de dependencias
!pip install rdkit-pypi==2022.9.5
!pip install gensim==4.3.2
!pip install torch==2.2.0
!pip install umap-learn==0.5.6
!pip install hdbscan==0.8.38
!pip install plotly==5.20.0
!pip install tqdm scikit-learn pandas numpy scipy

---
## 1. Descarga de datos y modelo

**Motivación:** Necesitamos dos recursos externos antes de poder comenzar:

1. **`model_300dim.pkl`** — El modelo preentrenado de mol2vec. Fue entrenado sobre millones de moléculas del corpus ZINC usando Word2Vec. Captura relaciones semánticas entre subestructuras moleculares (Morgan fingerprints) de forma análoga a como Word2Vec captura relaciones entre palabras.

2. **`qm9_dataset.csv`** — El dataset QM9 en formato tabular con la columna `smiles` que usaremos como punto de entrada.

Ambos se descargan una sola vez y se almacenan en sus respectivas carpetas del proyecto.

In [ ]:
import shutil

RUTA_MODELO = os.path.join(BASE_DIR, 'models', 'model_300dim.pkl')
RUTA_DATASET = os.path.join(BASE_DIR, 'data', 'qm9.csv')

# Solo descargamos si los archivos no existen ya
if not os.path.exists(RUTA_MODELO):
    print("Descargando modelo mol2vec preentrenado...")
    !wget -q https://raw.githubusercontent.com/samoturk/mol2vec/master/examples/models/model_300dim.pkl
    shutil.move('model_300dim.pkl', RUTA_MODELO)
    print(f"Modelo guardado en: {RUTA_MODELO}")
else:
    print(f"Modelo ya existe: {RUTA_MODELO}")

if not os.path.exists(RUTA_DATASET):
    print("Descargando dataset QM9...")
    !wget -q https://huggingface.co/datasets/n0w0f/qm9-csv/resolve/main/qm9_dataset.csv
    shutil.move('qm9_dataset.csv', RUTA_DATASET)
    print(f"Dataset guardado en: {RUTA_DATASET}")
else:
    print(f"Dataset ya existe: {RUTA_DATASET}")

---
## 2. Generación de embeddings moleculares con mol2vec

**Motivación:** Los algoritmos de machine learning no pueden operar directamente sobre cadenas SMILES. Es necesario convertir cada molécula en un vector numérico que capture su estructura química.

**¿Qué hace esta sección?**
1. Carga el dataset QM9 y el modelo mol2vec preentrenado.
2. Convierte cada SMILES a un objeto RDKit `Mol`.
3. Genera una "oración" molecular: una secuencia de tokens que representan los Morgan fingerprints de radio 1 alrededor de cada átomo.
4. Pasa esas oraciones por el modelo Word2Vec para obtener un vector de 300 dimensiones por molécula.

**¿Por qué mol2vec?**
mol2vec aplica la misma intuición que Word2Vec al dominio químico: subestructuras que aparecen en contextos moleculares similares tendrán representaciones vectoriales cercanas. El modelo de 300 dimensiones preentrenado sobre ZINC captura relaciones farmacofóricas y topológicas sin necesidad de descriptores manuales. Alternativamente se podrían usar Morgan fingerprints binarios, pero estos pierden información de contexto estructural que mol2vec sí preserva.

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from gensim.models import word2vec
from mol2vec.features import mol2alt_sentence, sentences2vec
from tqdm import tqdm

RUTA_EMBEDDINGS_CSV = os.path.join(BASE_DIR, 'data', 'qm9_embeddings_flat.csv')
RUTA_EMBEDDINGS_NPY = os.path.join(BASE_DIR, 'data', 'qm9_embeddings.npy')

if os.path.exists(RUTA_EMBEDDINGS_CSV) and os.path.exists(RUTA_EMBEDDINGS_NPY):
    print("Embeddings ya generados. Cargando desde disco...")
    df_final = pd.read_csv(RUTA_EMBEDDINGS_CSV)
    vectors_matrix = np.load(RUTA_EMBEDDINGS_NPY)
    print(f"Embeddings cargados: {vectors_matrix.shape} (moléculas × dimensiones)")
else:
    print("1. Cargando datos y modelo...")
    # Leemos únicamente la columna SMILES para no cargar propiedades innecesarias
    df = pd.read_csv(RUTA_DATASET, usecols=['smiles'])
    model_mol2vec = word2vec.Word2Vec.load(RUTA_MODELO)
    print(f"   Moléculas en dataset: {len(df)}")

    print("2. Convirtiendo SMILES a oraciones moleculares (Morgan r=1)...")
    tqdm.pandas()

    def get_sentence(smiles):
        # radius=1 genera tokens de vecindad inmediata de cada átomo.
        # Es el radio recomendado por los autores de mol2vec para este modelo.
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                return mol2alt_sentence(mol, radius=1)
            return None
        except:
            return None

    df['sentence'] = df['smiles'].progress_apply(get_sentence)
    df_clean = df.dropna(subset=['sentence']).reset_index(drop=True)
    print(f"   Moléculas procesadas correctamente: {len(df_clean)}")

    print("3. Generando vectores de 300 dimensiones...")
    # sentences2vec promedia los vectores Word2Vec de cada token de la oración.
    # Los tokens desconocidos (subestructuras no vistas en entrenamiento) se mapean a 'UNK'.
    vectors_matrix = sentences2vec(df_clean['sentence'].tolist(), model_mol2vec, unseen='UNK')
    np.save(RUTA_EMBEDDINGS_NPY, vectors_matrix)

    print("4. Construyendo y guardando tabla final (smiles + 300 columnas)...")
    col_names = [f'dim_{i}' for i in range(300)]
    df_vectors = pd.DataFrame(vectors_matrix, columns=col_names)
    df_final = pd.concat([df_clean[['smiles']], df_vectors], axis=1)
    df_final.to_csv(RUTA_EMBEDDINGS_CSV, index=False)

    print(f"\nArchivo generado: {RUTA_EMBEDDINGS_CSV}")
    print(f"Forma: {df_final.shape}  →  {len(df_final)} moléculas, 300 dimensiones + columna SMILES")
    print(df_final.head(3))

---
## 3. Búsqueda de similitud en el espacio completo (300D) con distancia de Mahalanobis

**Motivación:** Antes de comprimir el espacio, vale la pena explorar qué tan bien funciona la búsqueda de similitud directamente en los 300 embeddings. Esto nos da una línea base para comparar con la búsqueda en el espacio latente (sección 5).

**¿Qué hace esta sección?**
Dado un índice de molécula objetivo, calcula la distancia de Mahalanobis entre esa molécula y todas las demás, y devuelve las `top_n` más cercanas.

**¿Por qué Mahalanobis y no Euclidiana?**
La distancia Euclidiana trata todas las dimensiones como igualmente importantes e independientes. La distancia de Mahalanobis, en cambio, tiene en cuenta la correlación entre dimensiones y normaliza por la varianza de cada una. En espacios de alta dimensión con correlaciones entre features (como embeddings moleculares), Mahalanobis produce vecindades más coherentes semánticamente.

In [ ]:
import matplotlib.pyplot as plt
from rdkit import Chem
from scipy.spatial import distance

# Cargar el DataFrame con embeddings planos
df_embeddings = pd.read_csv(RUTA_EMBEDDINGS_CSV)

def obtener_smiles_similares_mahalanobis(df, indice_target, top_n=5):
    """
    Encuentra las 'top_n' moléculas más similares a la molécula en 'indice_target'
    usando distancia de Mahalanobis sobre los embeddings de 300 dimensiones.

    Parámetros:
    -----------
    df : DataFrame con columna 'smiles' y 300 columnas de dimensiones
    indice_target : índice de la molécula de referencia
    top_n : número de vecinos más cercanos a devolver

    Retorna:
    --------
    Lista de objetos Mol de RDKit de las moléculas más similares
    """
    # Vector de la molécula objetivo (excluimos la columna SMILES)
    fila_target = df.iloc[indice_target, 1:].values

    # Matriz de todos los embeddings
    matriz = df.iloc[:, 1:].values

    # cdist calcula todas las distancias de una sola vez (más eficiente que un loop)
    distancias = distance.cdist([fila_target], matriz, metric='mahalanobis')

    # Ordenamos ascendentemente y tomamos los top_n índices
    indices_cercanos = np.argsort(distancias[0])[:top_n]

    lista_smiles = df.iloc[indices_cercanos]['smiles'].tolist()
    mols = [Chem.MolFromSmiles(s) for s in lista_smiles]

    return [m for m in mols if m is not None]

In [ ]:
from rdkit.Chem import Draw

# Fijamos la semilla para reproducibilidad
np.random.seed(42)
indice_aleatorio = np.random.randint(0, len(df_embeddings))
print(f"Molécula objetivo: índice {indice_aleatorio}")
print(f"SMILES: {df_embeddings.iloc[indice_aleatorio]['smiles']}")

# Búsqueda en espacio de 300 dimensiones
mols_similares_300d = obtener_smiles_similares_mahalanobis(df_embeddings, indice_aleatorio, top_n=5)

def mostrar_moleculas(moleculas, titulo="Moléculas similares"):
    img = Draw.MolsToGridImage(
        moleculas,
        molsPerRow=5,
        subImgSize=(200, 200),
        legends=[Chem.MolToSmiles(m) for m in moleculas]
    )
    return img

print("\nTop 5 moléculas similares en espacio de 300D (Mahalanobis):")
mostrar_moleculas(mols_similares_300d)

---
## 4. Reducción de dimensionalidad con Autoencoder (300D → 32D)

**Motivación:** Los vectores de 300 dimensiones son funcionales, pero presentan dos problemas para el análisis posterior:
- **Maldición de la dimensionalidad**: En espacios de alta dimensión, las distancias pierden discriminación. Todos los puntos tienden a ser equidistantes entre sí.
- **Costo computacional**: Algoritmos como UMAP o HDBSCAN escalan mal con el número de dimensiones.

**¿Qué hace esta sección?**
Entrena un autoencoder con arquitectura simétrica encoder-decoder. El encoder comprime cada vector de 300D a 32D (espacio latente). El decoder intenta reconstruir los 300D originales a partir de esos 32D. El entrenamiento minimiza el error de reconstrucción (MSE), lo que fuerza al espacio latente a retener la información más relevante.

**¿Por qué un autoencoder y no PCA?**
PCA realiza una proyección **lineal**: no puede capturar relaciones no lineales entre dimensiones, que son habituales en espacios de embeddings moleculares. El autoencoder, al usar capas con activaciones no lineales (ReLU), puede aprender manifolds curvos en el espacio original. Esto produce una compresión más fiel a la estructura real del espacio químico.

**Arquitectura:**
```
Encoder: 300 → 128 → 64 → 32  (ReLU entre capas)
Decoder: 32  → 64  → 128 → 300 (ReLU entre capas)
```

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

RUTA_LATENTE = os.path.join(BASE_DIR, 'data', 'qm9_latent.npy')

# Cargar embeddings crudos
X = np.load(RUTA_EMBEDDINGS_NPY)
print(f"Embeddings cargados: {X.shape}")

# Normalización: StandardScaler lleva cada dimensión a media=0, std=1.
# Es imprescindible antes de entrenar redes neuronales para que el gradiente
# no esté dominado por dimensiones con mayor escala numérica.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Normalización completada (media=0, std=1 por dimensión)")

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=300, latent_dim=32):
        super().__init__()

        # El encoder va reduciendo progresivamente la dimensión.
        # Un cuello de botella gradual (300→128→64→32) en lugar de un salto
        # directo (300→32) permite al modelo aprender representaciones intermedias
        # y produce una compresión más suave y generalizable.
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )

        # El decoder es simétrico: debe aprender a reconstruir la información
        # que el encoder comprimió. La simetría no es estrictamente necesaria,
        # pero es una buena práctica que facilita el entrenamiento estable.
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)       # z es el vector latente de 32D
        recon = self.decoder(z)   # reconstrucción de los 300D originales
        return recon, z


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de entrenamiento: {device}")

In [ ]:
if os.path.exists(RUTA_LATENTE):
    print("Espacio latente ya calculado. Cargando desde disco...")
    latent_vectors = np.load(RUTA_LATENTE)
    print(f"Vectores latentes cargados: {latent_vectors.shape}")
else:
    print("Entrenando autoencoder...")
    model_ae = Autoencoder().to(device)

    # Adam con lr=1e-3 es el estándar para autoencoders de este tamaño.
    # MSELoss mide el error de reconstrucción cuadrático medio, apropiado
    # para espacios continuos como embeddings vectoriales.
    optimizer = torch.optim.Adam(model_ae.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)

    epochs = 10000
    for epoch in range(epochs):
        optimizer.zero_grad()
        recon, _ = model_ae(X_tensor)
        loss = criterion(recon, X_tensor)
        loss.backward()
        optimizer.step()

        if epoch % 1000 == 0:
            print(f"  Época {epoch:5d} / {epochs}  |  Loss: {loss.item():.6f}")

    # Extraemos los vectores latentes en modo evaluación
    # (desactiva dropout/batchnorm si los hubiera)
    model_ae.eval()
    with torch.no_grad():
        _, latent = model_ae(X_tensor)

    latent_vectors = latent.cpu().numpy()
    np.save(RUTA_LATENTE, latent_vectors)
    print(f"\nEspacio latente guardado: {latent_vectors.shape}")

---
## 5. Búsqueda de similitud en el espacio latente (32D) con Mahalanobis

**Motivación:** Ahora repetimos la misma búsqueda de similitud de la sección 3, pero sobre el espacio comprimido de 32 dimensiones. El objetivo es verificar si el autoencoder preservó la estructura de vecindad del espacio original: ¿las moléculas similares en 300D siguen siendo similares en 32D?

**¿Qué hace esta sección?**
Construye un DataFrame con los vectores latentes de 32D y aplica la misma función de búsqueda por Mahalanobis sobre el mismo índice objetivo que se usó en la sección 3, para que los resultados sean comparables directamente.

In [ ]:
# Construir DataFrame del espacio latente en el mismo formato que df_embeddings
# (primera columna: smiles, resto: dimensiones del espacio latente)
df_latente = pd.DataFrame(latent_vectors)
df_latente.insert(0, 'smiles', df_embeddings['smiles'].values)

print(f"Espacio latente: {df_latente.shape}  →  {latent_vectors.shape[0]} moléculas, {latent_vectors.shape[1]} dimensiones")

# Búsqueda en espacio de 32 dimensiones (mismo índice que en sección 3)
mols_similares_32d = obtener_smiles_similares_mahalanobis(df_latente, indice_aleatorio, top_n=5)

print(f"\nTop 5 moléculas similares en espacio latente 32D (Mahalanobis):")
print("(Compara visualmente con los resultados de la sección 3)")
mostrar_moleculas(mols_similares_32d)

---
## 6. Descomposición de Cholesky: optimización del espacio para búsquedas rápidas

**Motivación:** La distancia de Mahalanobis es estadísticamente correcta, pero tiene un costo computacional importante: requiere calcular y almacenar la matriz de covarianza inversa (32×32) y realizar una multiplicación matricial por cada consulta. Para 133,000 moléculas esto se vuelve un cuello de botella.

**¿Qué hace esta sección?**
Aplica la descomposición de Cholesky a la matriz de covarianza inversa para encontrar una transformación lineal del espacio tal que, en el espacio transformado, la **distancia Euclidiana sea equivalente a la Mahalanobis original**.

**¿Por qué funciona esto?**
Si `VI` es la matriz de covarianza inversa y `L` es su factor de Cholesky (tal que `L · Lᵀ = VI`), entonces para cualquier par de vectores `a` y `b`:
```
Mahalanobis(a, b)² = (a-b)ᵀ · VI · (a-b)
                   = (a-b)ᵀ · L · Lᵀ · (a-b)
                   = ||Lᵀ(a-b)||²
                   = Euclidiana(La, Lb)²
```
Al transformar todos los vectores como `x' = L · x`, podemos usar distancia Euclidiana (y beneficiarnos de estructuras como KD-Tree o ball-tree) en lugar de Mahalanobis para cualquier búsqueda futura.

In [ ]:
from scipy.linalg import cholesky

RUTA_CHOLESKY = os.path.join(BASE_DIR, 'data', 'qm9_latent_mahalanobis_space.npy')

if os.path.exists(RUTA_CHOLESKY):
    print("Espacio transformado ya existe. Cargando desde disco...")
    X_transformado = np.load(RUTA_CHOLESKY)
    print(f"Espacio transformado cargado: {X_transformado.shape}")
else:
    print(f"1. Espacio latente: {latent_vectors.shape}  ({latent_vectors.nbytes / 1e6:.1f} MB)")

    print("2. Calculando matriz de covarianza inversa (VI)...")
    # rowvar=False: cada fila es una muestra, cada columna una variable
    cov_matrix = np.cov(latent_vectors, rowvar=False)
    VI = np.linalg.inv(cov_matrix)

    print("3. Descomposición de Cholesky de VI...")
    # lower=True: devuelve la matriz triangular inferior L tal que L · Lᵀ = VI
    L = cholesky(VI, lower=True)

    print("4. Transformando el espacio latente (x' = L · x)...")
    # Ahora ||x'_a - x'_b||_2 = Mahalanobis(x_a, x_b) en el espacio original
    X_transformado = np.dot(latent_vectors, L)

    print("5. Guardando espacio optimizado...")
    np.save(RUTA_CHOLESKY, X_transformado)
    print(f"Listo. Espacio transformado: {X_transformado.shape}  ({X_transformado.nbytes / 1e6:.1f} MB)")

---
## 7. Chupón Gaussiano: vecindad topológica ponderada

**Motivación:** La búsqueda de vecinos más cercanos devuelve una lista discreta de moléculas. En ocasiones es más útil tener una **función de similitud continua**: una puntuación que vaya de 1.0 (idéntica al objetivo) a 0.0 (completamente distante), sin un corte arbitrario. Esto permite visualizar la "zona de influencia" de una molécula sobre el mapa topológico.

**¿Qué hace esta sección?**
Dada una molécula objetivo, calcula para cada molécula del dataset un peso gaussiano:
```
peso(x) = exp(-d(x, x₀)² / σ²)
```
donde `d` es la distancia Euclidiana en el espacio transformado por Cholesky (equivalente a Mahalanobis en el espacio latente) y `σ` controla el ancho del "chupón".

**¿Por qué una campana de Gauss?**
La función gaussiana tiene propiedades matemáticas deseables: es suave, diferenciable, monotónicamente decreciente con la distancia, y decae rápidamente lejos del centro. A diferencia de un radio fijo (que produce un corte abrupto), el chupón gaussiano asigna pesos que reflejan gradualmente la similitud estructural. El parámetro `sigma` y el `umbral` permiten controlar el radio efectivo de la búsqueda.

In [ ]:
from scipy.spatial.distance import cdist

# Cargar los SMILES del dataset para asociar índices con estructuras
df_smiles = pd.read_csv(RUTA_EMBEDDINGS_CSV, usecols=['smiles'])

def obtener_chupon_gaussiano(X, df_smiles, idx_target, sigma=2.0, umbral=0.1):
    """
    Aplica una función gaussiana centrada en la molécula objetivo para ponderar
    la similitud de todas las moléculas del dataset.

    Parámetros:
    -----------
    X : array (N, D) — espacio transformado por Cholesky
    df_smiles : DataFrame con columna 'smiles'
    idx_target : índice de la molécula objetivo
    sigma : ancho del chupón (mayor sigma = vecindad más amplia)
    umbral : peso mínimo para incluir en los resultados (entre 0 y 1)

    Retorna:
    --------
    DataFrame con columnas: indice_original, smiles, peso_topologico
    """
    x0 = X[idx_target]

    # Distancia al cuadrado de x0 a todos los demás vectores.
    # Operación vectorizada: evita loops y es órdenes de magnitud más rápida.
    distancias_cuadradas = np.sum((X - x0) ** 2, axis=1)

    # Función campana de Gauss centrada en x0
    pesos = np.exp(-distancias_cuadradas / (sigma ** 2))

    # Filtrar por umbral: solo moléculas con similaridad >= umbral
    indices_dentro = np.where(pesos >= umbral)[0]

    resultados = pd.DataFrame({
        'indice_original': indices_dentro,
        'smiles': df_smiles.iloc[indices_dentro]['smiles'].values,
        'peso_topologico': pesos[indices_dentro]
    })

    return resultados.sort_values('peso_topologico', ascending=False).reset_index(drop=True)

In [ ]:
# Usamos el mismo índice aleatorio para mantener coherencia con secciones anteriores
indice_target = indice_aleatorio

# sigma=5.0: vecindad moderada en el espacio transformado.
# umbral=0.1: incluimos moléculas con al menos 10% de similitud al objetivo.
df_chupon = obtener_chupon_gaussiano(X_transformado, df_smiles, indice_target, sigma=5.0, umbral=0.1)

print(f"El chupón capturó {len(df_chupon)} moléculas")
print("\nTop 10 moléculas en el centro del chupón:")
print(df_chupon.head(10).to_string(index=False))

In [ ]:
# Visualización de las 10 moléculas más cercanas al centro del chupón
top_chupon = df_chupon.head(10)
mols_chupon = [Chem.MolFromSmiles(s) for s in top_chupon['smiles']]
leyendas = [f"Peso: {p:.3f}" for p in top_chupon['peso_topologico']]

imagen_chupon = Draw.MolsToGridImage(
    mols_chupon,
    molsPerRow=5,
    subImgSize=(250, 250),
    legends=leyendas
)
imagen_chupon

---
## 8. Mapas topológicos: UMAP + HDBSCAN

**Motivación:** El espacio latente de 32 dimensiones (incluso tras la transformación de Cholesky) no es directamente visualizable. Para explorar la organización global del espacio químico necesitamos proyectarlo a 2 dimensiones conservando la estructura de vecindad local.

**¿Qué hace esta sección?**
- **UMAP** proyecta los 32D a 2D preservando estructura local (vecinos cercanos) y parte de la estructura global (relaciones entre grupos).
- **HDBSCAN** sobre las coordenadas 2D de UMAP identifica automáticamente agrupaciones de moléculas sin necesidad de especificar el número de clusters a priori. Las moléculas que no encajan en ningún cluster se etiquetan como ruido (-1).

**¿Por qué UMAP sobre t-SNE para el clustering?**
UMAP preserva mejor la estructura global del espacio (no solo la local), lo que produce clusters más coherentes para HDBSCAN. t-SNE es excelente para visualización pero distorsiona las distancias entre clusters, haciendo que HDBSCAN sobre t-SNE produzca agrupaciones menos confiables. Por eso HDBSCAN se aplica sobre UMAP en ambas variantes.

**Parámetros clave de HDBSCAN:**
- `min_cluster_size=500`: un grupo debe contener al menos 500 moléculas para considerarse una "familia química" válida.
- `min_samples=50`: controla la robustez frente al ruido; valores más altos generan más puntos de ruido pero clusters más densos.

In [ ]:
import umap
import hdbscan
import plotly.express as px

RUTA_UMAP = os.path.join(BASE_DIR, 'data', 'qm9_umap_2d.npy')

# UMAP es costoso (~2 min para 133k puntos). Lo guardamos para no recalcular.
if os.path.exists(RUTA_UMAP):
    print("Coordenadas UMAP ya calculadas. Cargando desde disco...")
    X_umap = np.load(RUTA_UMAP)
else:
    print("Calculando UMAP (esto tomará ~2 minutos)...")
    # n_neighbors=15: equilibrio entre preservación local y global
    # min_dist=0.1: permite clusters compactos con separación visible
    # metric='euclidean': correcto porque el espacio ya fue transformado con Cholesky
    reductor_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean', random_state=42)
    X_umap = reductor_umap.fit_transform(X_transformado)
    np.save(RUTA_UMAP, X_umap)
    print(f"UMAP guardado: {RUTA_UMAP}")

print(f"Coordenadas UMAP: {X_umap.shape}")

# Clustering con HDBSCAN sobre el mapa 2D de UMAP
print("Aplicando HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=500, min_samples=50, metric='euclidean')
labels_hdbscan = clusterer.fit_predict(X_umap)
n_clusters = len(set(labels_hdbscan)) - (1 if -1 in labels_hdbscan else 0)
print(f"HDBSCAN encontró {n_clusters} familias/clusters distintos")
print(f"Moléculas clasificadas como ruido: {(labels_hdbscan == -1).sum()}")

In [ ]:
# Construir DataFrame para visualización
df_mapa_umap = pd.DataFrame(X_umap, columns=['UMAP_1', 'UMAP_2'])
df_mapa_umap['SMILES'] = df_smiles['smiles']
df_mapa_umap['Cluster'] = [f"Cluster {x}" if x != -1 else "Ruido (-1)" for x in labels_hdbscan]

# Ordenar para que el ruido se dibuje al fondo (primero) y los clusters encima
df_mapa_umap = df_mapa_umap.sort_values('Cluster')

fig_umap = px.scatter(
    df_mapa_umap,
    x='UMAP_1', y='UMAP_2',
    color='Cluster',
    hover_data=['SMILES'],
    title="Topología de QM9: UMAP + HDBSCAN",
    render_mode='webgl',  # Necesario para 133k puntos sin lag
    color_discrete_sequence=px.colors.qualitative.Alphabet
)
fig_umap.update_traces(marker=dict(size=2, opacity=0.7))

RUTA_HTML_UMAP = os.path.join(BASE_DIR, 'html', 'mapa_topologico_HDBSCAN_UMAP.html')
fig_umap.write_html(RUTA_HTML_UMAP)
print(f"Mapa guardado en: {RUTA_HTML_UMAP}")
fig_umap.show()

---
## 9. Mapa topológico: t-SNE + HDBSCAN

**Motivación:** t-SNE es un algoritmo de reducción de dimensionalidad complementario a UMAP. A diferencia de UMAP, t-SNE maximiza la separación visual entre grupos a costa de distorsionar las distancias entre clusters. Esto lo hace especialmente útil para identificar visualmente estructuras finas dentro de cada agrupación.

**¿Qué hace esta sección?**
Genera una proyección 2D alternativa con t-SNE y aplica el mismo HDBSCAN (entrenado sobre UMAP) para colorear los puntos por cluster. Esto permite comparar cómo luce la misma estructura de clusters bajo ambas proyecciones.

**Nota sobre el tiempo de cómputo:** t-SNE escala cuadráticamente con el número de puntos. Para ~133k moléculas puede tardar entre 10 y 30 minutos. Se guarda en disco para evitar recalcular.

In [ ]:
from sklearn.manifold import TSNE

RUTA_TSNE = os.path.join(BASE_DIR, 'data', 'qm9_tsne_2d.npy')

if os.path.exists(RUTA_TSNE):
    print("Coordenadas t-SNE ya calculadas. Cargando desde disco...")
    X_tsne = np.load(RUTA_TSNE)
else:
    print("Calculando t-SNE (puede tardar entre 10 y 30 minutos)...")
    # perplexity=30: balance entre estructura local y global (valor estándar)
    # n_jobs=-1: usa todos los núcleos disponibles para acelerar el cálculo
    X_tsne = TSNE(n_components=2, perplexity=30, n_jobs=-1, random_state=42).fit_transform(X_transformado)
    np.save(RUTA_TSNE, X_tsne)
    print(f"t-SNE guardado: {RUTA_TSNE}")

print(f"Coordenadas t-SNE: {X_tsne.shape}")

# Usamos los mismos labels de HDBSCAN calculados sobre UMAP (sección 8)
# para comparar la distribución de clusters en la proyección t-SNE
df_mapa_tsne = pd.DataFrame(X_tsne, columns=['TSNE_1', 'TSNE_2'])
df_mapa_tsne['SMILES'] = df_smiles['smiles']
df_mapa_tsne['Cluster'] = [f"Cluster {x}" if x != -1 else "Ruido (-1)" for x in labels_hdbscan]
df_mapa_tsne = df_mapa_tsne.sort_values('Cluster')

fig_tsne = px.scatter(
    df_mapa_tsne,
    x='TSNE_1', y='TSNE_2',
    color='Cluster',
    hover_data=['SMILES'],
    title="Topología de QM9: t-SNE (proyección) + HDBSCAN (clusters de UMAP)",
    render_mode='webgl',
    color_discrete_sequence=px.colors.qualitative.Alphabet
)
fig_tsne.update_traces(marker=dict(size=2, opacity=0.7))

RUTA_HTML_TSNE = os.path.join(BASE_DIR, 'html', 'mapa_topologico_HDBSCAN_T-SNE.html')
fig_tsne.write_html(RUTA_HTML_TSNE)
print(f"Mapa guardado en: {RUTA_HTML_TSNE}")
fig_tsne.show()

---
## 10. Mapa topológico con Chupón Gaussiano

**Motivación:** Hasta ahora hemos visto el mapa topológico coloreado por cluster (visión global). Esta sección complementa esa vista con una perspectiva local: colorear el mapa según el peso gaussiano respecto a una molécula objetivo, para visualizar exactamente dónde se ubica su vecindad en el espacio químico.

**¿Qué hace esta sección?**
Proyecta el chupón gaussiano (calculado en sección 7) sobre los mapas 2D de UMAP y t-SNE. Los puntos más cercanos a la molécula objetivo aparecen en blanco/amarillo y los más lejanos en negro, usando una escala de color "caliente" (hot).

**¿Qué nos dice esta visualización?**
Permite identificar si la vecindad de una molécula es compacta (cluster bien definido) o dispersa (estructura que se extiende entre varios clusters). Es una herramienta de exploración interactiva: cambiando `indice_target` y `sigma` se puede navegar el espacio químico de forma intuitiva.

In [ ]:
# Chupón gaussiano sobre UMAP
# X_transformado ya está en memoria desde la sección 6
df_mapa_chupon_umap = pd.DataFrame(X_umap, columns=['UMAP_1', 'UMAP_2'])
df_mapa_chupon_umap['SMILES'] = df_smiles['smiles']
df_mapa_chupon_umap['Peso_Topologico'] = 0.0

df_chupon_umap = obtener_chupon_gaussiano(X_transformado, df_smiles, indice_target, sigma=5.0, umbral=0.1)

# Asignar pesos gaussianos a los índices que entraron en el chupón
df_mapa_chupon_umap.loc[df_chupon_umap['indice_original'], 'Peso_Topologico'] = df_chupon_umap['peso_topologico'].values
df_mapa_chupon_umap = df_mapa_chupon_umap.sort_values('Peso_Topologico')

fig_chupon_umap = px.scatter(
    df_mapa_chupon_umap,
    x='UMAP_1', y='UMAP_2',
    color='Peso_Topologico',
    hover_data=['SMILES'],
    color_continuous_scale='hot',
    title=f"Mapa Topológico UMAP: Chupón Gaussiano (Target idx: {indice_target})",
    render_mode='webgl'
)
fig_chupon_umap.update_traces(marker=dict(size=3, opacity=0.7))

RUTA_HTML_CHUPON_UMAP = os.path.join(BASE_DIR, 'html', 'mapa_chupon_umap.html')
fig_chupon_umap.write_html(RUTA_HTML_CHUPON_UMAP)
print(f"Mapa guardado en: {RUTA_HTML_CHUPON_UMAP}")
fig_chupon_umap.show()

In [ ]:
# Chupón gaussiano sobre t-SNE
df_mapa_chupon_tsne = pd.DataFrame(X_tsne, columns=['TSNE_1', 'TSNE_2'])
df_mapa_chupon_tsne['SMILES'] = df_smiles['smiles']
df_mapa_chupon_tsne['Peso_Topologico'] = 0.0

df_chupon_tsne = obtener_chupon_gaussiano(X_transformado, df_smiles, indice_target, sigma=5.0, umbral=0.1)

df_mapa_chupon_tsne.loc[df_chupon_tsne['indice_original'], 'Peso_Topologico'] = df_chupon_tsne['peso_topologico'].values
df_mapa_chupon_tsne = df_mapa_chupon_tsne.sort_values('Peso_Topologico')

fig_chupon_tsne = px.scatter(
    df_mapa_chupon_tsne,
    x='TSNE_1', y='TSNE_2',
    color='Peso_Topologico',
    hover_data=['SMILES'],
    color_continuous_scale='hot',
    title=f"Mapa Topológico t-SNE: Chupón Gaussiano (Target idx: {indice_target})",
    render_mode='webgl'
)
fig_chupon_tsne.update_traces(marker=dict(size=3, opacity=0.7))

RUTA_HTML_CHUPON_TSNE = os.path.join(BASE_DIR, 'html', 'mapa_chupon_tsne.html')
fig_chupon_tsne.write_html(RUTA_HTML_CHUPON_TSNE)
print(f"Mapa guardado en: {RUTA_HTML_CHUPON_TSNE}")
fig_chupon_tsne.show()